In [2]:
# 1. Definisikan Gejala (Evidence)
gejala = {
'G01': 'Ruam Kulit Kemerahan',
'G02': 'Gatal',
'G03': 'Tekstur kulit kering',
'G04': 'Adanya pembengkakan',
'G05': 'Kulit bersisik',
'G06': 'Kulit melepuh',
'G07': 'Penebalan pada kulit',
'G08': 'Kulit pecah-pecah',
'G09': 'Kulit terasa nyeri atau sakit',
'G10': 'Penyakit dapat menyebar',
'G11': 'Luka yang mengeluarkan cairan',
'G12': 'Bentuk ruam tidak beraturan',
'G13': 'Adanya bercak putih, coklat, merah',
'G14': 'Terdapat bintil-bintil kecil',
'G15': 'Bentuk melingkar seperti cincin',
'G16': 'Demam',
'G17': 'Pilek',
'G18': 'Cepat merasa lelah',
'G19': 'Nyeri sendi',
'G20': 'Pusing dan sakit kepala',
'G21': 'Kulit menjadi sensitif'
}
# 2. Definisikan Penyakit (Hipotesis)
penyakit = {
'P01': 'Dermatitis Alergi',
'P02': 'Dermatitis Atopik',
'P03': 'Urtikaria',
'P04': 'Panu',
'P05': 'Tinea/Kurap',
'P06': 'Herpes Zooster',
'P07': 'Biang Keringat'
}
# 3. Definisikan Aturan (Rules)
# Ini adalah daftar gejala yang berhubungan dengan tiap penyakit
rules = {
'P01': ['G01', 'G02', 'G03', 'G04', 'G05', 'G06', 'G07', 'G08', 'G09',
'G11'],
'P02': ['G01', 'G02', 'G03', 'G04', 'G05', 'G06', 'G09', 'G10', 'G11'],
'P03': ['G01', 'G02', 'G07', 'G09', 'G10', 'G12'],
'P04': ['G02', 'G05', 'G10', 'G13'],
'P05': ['G02', 'G05', 'G14', 'G15'],
'P06': ['G06', 'G09', 'G11', 'G16', 'G17', 'G18', 'G19', 'G20', 'G21'],
'P07': ['G01', 'G02', 'G14']
}
# 4. Definisikan Bobot MB dan MD
# Format: (Penyakit, Gejala): (MB, MD)
cf_weights = {
# Dermatitis Alergi (P01)
('P01', 'G01'): (0.8, 0.2),
('P01', 'G02'): (0.6, 0.4),
('P01', 'G03'): (0.8, 0.2),
('P01', 'G04'): (0.4, 0.6),
('P01', 'G05'): (0.6, 0.4),
('P01', 'G06'): (0.2, 0.8),
('P01', 'G07'): (0.4, 0.6),
('P01', 'G08'): (0.4, 0.6),
('P01', 'G09'): (0.6, 0.4),
('P01', 'G11'): (0.4, 0.6),
# Urtikaria (P03) [cite: 453]
('P03', 'G01'): (0.8, 0.2),
('P03', 'G02'): (0.8, 0.2),
('P03', 'G07'): (0.8, 0.2),
('P03', 'G09'): (0.4, 0.6),
('P03', 'G10'): (0.8, 0.2),
('P03', 'G12'): (0.6, 0.4),
# Panu (P04) [cite: 453]
('P04', 'G02'): (0.8, 0.2),
('P04', 'G05'): (0.8, 0.2),
('P04', 'G10'): (0.6, 0.4),
('P04', 'G13'): (0.8, 0.2),
# Tinea/Kurap (P05) [cite: 477]
('P05', 'G02'): (0.8, 0.2),
('P05', 'G05'): (0.8, 0.2),
('P05', 'G14'): (0.4, 0.6),
('P05', 'G15'): (0.8, 0.2), # Asumsi dari tabel hlm 20 (Penyakitberbentuk melingkar)
# Biang Keringat (P07) [cite: 478]
('P07', 'G01'): (0.6, 0.4),
('P07', 'G02'): (0.8, 0.2),
('P07', 'G14'): (0.8, 0.2),
# ... Lanjutkan untuk P02 dan P06 ...
# (Tugas mahasiswa untuk melengkapi)
}


In [4]:
def calculate_cf(mb, md):
    """Menghitung CF tunggal dari MB dan MD."""
    return mb - md


def combine_cf(cf1, cf2):
    """Mengkombinasikan dua nilai CF."""
    if cf1 >= 0 and cf2 >= 0:
        return cf1 + cf2 * (1 - cf1)
    elif cf1 < 0 and cf2 < 0:
        # Tambahan dari hlm. 7 [cite: 170]
        return cf1 + cf2 * (1 + cf1)
    else:
        # Tambahan dari hlm. 7 [cite: 163]
        return (cf1 + cf2) / (1 - min(abs(cf1), abs(cf2)))


In [9]:
def run_inference(gejala_pasien):
    """
    Menjalankan mesin inferensi untuk menghitung CF setiap penyakit
    berdasarkan gejala yang dialami pasien.
    """
    hasil_diagnosis = {}

    # Iterasi setiap penyakit dalam basis pengetahuan
    for kode_penyakit, nama_penyakit in penyakit.items():

        # Dapatkan daftar gejala yang relevan
        gejala_relevan = rules.get(kode_penyakit, [])

        # Filter gejala pasien yang cocok
        gejala_cocok = [g for g in gejala_pasien if g in gejala_relevan]

        if not gejala_cocok:
            continue

        # Hitung CF untuk setiap gejala yang cocok
        cf_list = []
        for g in gejala_cocok:
            mb, md = cf_weights.get((kode_penyakit, g), (0, 0))

            if mb + md > 0:  # ada bobot
                cf = calculate_cf(mb, md)
                cf_list.append(cf)

        if not cf_list:
            continue

        # Kombinasi CF
        cf_final = cf_list[0]
        for i in range(1, len(cf_list)):
            cf_final = combine_cf(cf_final, cf_list[i])

        # Simpan
        hasil_diagnosis[nama_penyakit] = cf_final

    # Urutkan hasil
    hasil_urut = sorted(
        hasil_diagnosis.items(),
        key=lambda item: item[1],
        reverse=True
    )

    return hasil_urut


In [10]:
# Input gejala untuk Responden 179
input_179 = ['G02', 'G05', 'G01', 'G15']

# Jalankan inferensi
hasil_179 = run_inference(input_179)

# Tampilkan hasil
print("--- Hasil Diagnosis Responden 179 ---")
for penyakit, cf in hasil_179:
    print(f"{penyakit}: {cf * 100:.2f}%")


--- Hasil Diagnosis Responden 179 ---
Tinea/Kurap: 93.60%
Urtikaria: 84.00%
Panu: 84.00%
Dermatitis Alergi: 74.40%
Biang Keringat: 68.00%
